# Создание модели предсказания зарплат

Основная цель данного ноутбука — создать модель, которая сможет предсказывать зарплату от ряда наиболее значимых признаков.

Все необходимые данные и гипотезы уже изучены в [`hypothesis_testing.ipynb`](hypothesis_testing.ipynb): подготовка данных, 21 проверенная гипотеза, какие признаки реально полезны. Здесь мы используем выводы исследовательского ноутбука для построения модели прогноза.

**Задача:** по наблюдаемым признакам вакансии предсказывать `salary_mid` — среднюю зарплату в USD в год.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from src.utils import load_data

df = load_data()
df_original = df.copy()

print(f"Размерность датасета {df.shape}")
print(f"Количество дубликатов {df.duplicated().sum()}")

display(df.head())

## 1. Подготовка данных

Приводим зарплатные вилки к USD в год и считаем целевую переменную — середину вилки (`salary_mid`) и её логарифм (`log_salary`).

In [ ]:
from src.preprocessing import prepare_salary_data

df = prepare_salary_data(df)

display(
    df[
        [
            "salary_min",
            "salary_max",
            "salary_mid",
            "log_salary",
            "raw_period",
            "raw_currency",
        ]
    ].head()
)

## 2. Таргет и признаки

Исключаем строки с неполной вилкой (только одна граница известна) и отбрасываем колонки, которые не должны уходить в модель: целевой таргет, его логарифм, годовые суммы и текстовое описание зарплаты.

In [ ]:
df_model = df.loc[df["salary_mid"].notna()].copy()

X = df_model.drop(
    columns=[
        "salary_min",
        "salary_max",
        "salary_mid",
        "log_salary",
        "salary_min_annual",
        "salary_max_annual",
        "salary_text",
    ],
    errors="ignore",
)

y = df_model["log_salary"]

print(f"Размер X: {X.shape}")
print(f"Размер y: {y.shape}")
print(f"Пропуски в y: {y.isna().sum()}")

display(y.describe())

## 3. Разделение данных

Отдельно обучающая и проверочная выборки. Фиксированный `random_state` для воспроизводимости.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

## 4. Базовая модель

Оцениваем, с чего начинаем: медиану логарифмической зарплаты в качестве предиктора.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, y_pred_baseline)

print(f"Baseline MAE: {baseline_mae:.4f}")

## 5. Линейная модель

`Pipeline` с `ColumnTransformer`: числовые признаки — медианный импут + стандартизация, категориальные — импут модой + one-hot (`handle_unknown="ignore"`).

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = [column for column in X_train.columns if column not in numeric_features]

numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"Linear Regression MAE: {mae:.4f}")

## 6. Random Forest

Ансамбль деревьев на том же препроцессоре. Сравниваем с линейной моделью.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline( 
    [
        ("preprocessor", clone(preprocessor)),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=200,
                min_samples_leaf=3,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, y_pred_rf)

print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"Linear Regression MAE: {mae:.4f}")
print(f"Random Forest MAE: {rf_mae:.4f}")

## 7. Градиентный бустинг: HistGradientBoosting

Градиентный бустинг на табличных данных: деревья, каждое следующее исправляет ошибки предыдущих.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

categorical_pipeline_dense = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor_dense = ColumnTransformer(
    [
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline_dense, categorical_features),
    ]
)

hgb_model = Pipeline(
    [
        ("preprocessor", preprocessor_dense),
        (
            "regressor",
            HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.05,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                random_state=42,
            ),
        ),
    ]
)

hgb_model.fit(X_train, y_train)

y_pred_hgb = hgb_model.predict(X_test)
hgb_mae = mean_absolute_error(y_test, y_pred_hgb)

print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"Linear Regression MAE: {mae:.4f}")
print(f"Random Forest MAE: {rf_mae:.4f}")
print(f"HistGradientBoosting MAE: {hgb_mae:.4f}")

## 8. Метрики в USD

Логарифмическая ошибка возвращается в исходную шкалу через `exp`. Для бизнеса удобнее видеть MAE в долларах и MAPE (средняя ошибка в процентах).

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

y_test_usd = np.exp(y_test)
y_pred_usd = np.exp(y_pred)

mae_usd = mean_absolute_error(df_model.loc[X_test.index, "salary_mid"], y_pred_usd)
mape = mean_absolute_percentage_error(df_model.loc[X_test.index, "salary_mid"], y_pred_usd)

print(f"MAE в логарифмической шкале: {mae:.4f}")
print(f"MAE в USD: {mae_usd:,.2f}")
print(f"MAPE: {mape:.2%}")

## 9. Сравнение: модель на логарифмической шкале и на USD

Тот же набор признаков, но разная целевая переменная: логарифм (`log_salary`) против исходной суммы (`salary_mid`).

In [ ]:
linear_usd_model = Pipeline(
    [
        ("preprocessor", clone(preprocessor)),
        ("regressor", LinearRegression()),
    ]
)

y_train_usd = df_model.loc[X_train.index, "salary_mid"]
y_test_usd = df_model.loc[X_test.index, "salary_mid"]

linear_usd_model.fit(X_train, y_train_usd)

y_pred_usd_direct = linear_usd_model.predict(X_test)

mae_usd_direct = mean_absolute_error(y_test_usd, y_pred_usd_direct)
mape_usd_direct = mean_absolute_percentage_error(y_test_usd, y_pred_usd_direct)

print(f"Linear Regression — log target: MAE = {mae_usd:,.2f} USD")
print(f"Linear Regression — USD target: MAE = {mae_usd_direct:,.2f} USD")
print(f"MAPE при обучении на USD: {mape_usd_direct:.2%}")

## 10. Итоги

Сводка метрик всех моделей на тестовой выборке и вывод о лучшей модели.

In [ ]:
final_results = {}

for name, trained in [
    ("baseline (median)", baseline),
    ("linear_regression", model),
    ("random_forest", rf_model),
    ("hist_gradient_boosting", hgb_model),
]:
    pred_log = trained.predict(X_test)
    pred_usd = np.exp(pred_log)
    final_results[name] = {
        "MAE (log)": mean_absolute_error(y_test, pred_log),
        "MAE (USD)": float(mean_absolute_error(df_model.loc[X_test.index, "salary_mid"], pred_usd)),
        "MAPE": float(mean_absolute_percentage_error(df_model.loc[X_test.index, "salary_mid"], pred_usd)),
    }

comparison = pd.DataFrame(final_results).T
display(comparison)

best_model = comparison["MAE (log)"].idxmin()
print(f"Лучшая модель на тестовой выборке: {best_model}")